# Baseline on Colab L4: tune / holdout references
Open on colab.research.google.com; **Runtime -> Change runtime type -> L4 GPU**. See `colab/SETUP.md`. Never paste the HF token into a cell.

Reference sets (3 parquet files spread over the dataset): **tune** = `train-00000` (all tuning happens here),
**holdout** = `train-00024` + `train-00049` (scored only, never tuned on). `DRIVE=/content/drive/MyDrive/vidx`.
Run the cells in order. After a runtime reset redo steps 1-4, re-download the 3 reference files (step 5), recreate the symlinks (step 6) and, if the 50-file download (step 8) had not finished, start it again; then re-run the interrupted step.

**Brute-force reference runs pass `--sample 0 --precision fp32` explicitly:** since 2026-11-20 the `run.py` defaults are the tuned setting (k=3, fp16).

## 1. Machine (note for HARDWARE.md)

In [ ]:
!nvidia-smi; df -h /content; free -h; nproc; lscpu | grep "Model name"; python3 --version

## 2. Python 3.11 venv, code, model

In [ ]:
%%bash
set -e
cd /content
pip install -q uv
[ -d /content/venv ] || uv venv --python 3.11 /content/venv
[ -d video-indexing-yolo11 ] || git clone https://github.com/sobanali256/video-indexing-yolo11.git
cd video-indexing-yolo11 && git pull
apt-get install -y -q libgl1 libglib2.0-0
uv pip install --python /content/venv/bin/python -r requirements.txt
/content/venv/bin/python -c "import vidx,torch,av,ultralytics;print(torch.__version__,torch.cuda.is_available(),av.__version__,ultralytics.__version__)"
/content/venv/bin/python prepare.py

## 3. Hugging Face login (Colab Secret `HF_TOKEN`, else prompt)

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except Exception:
    import getpass
    login(getpass.getpass("HF token: "))

## 4. Google Drive (checkpoints, dumps, reference copies). If the mount fails use `--checkpoint-hf-repo` instead of `--checkpoint-dir` below

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 5. Download only the 3 reference files (tune 00000, holdout 00024 + 00049)

In [ ]:
!/content/venv/bin/hf download HuggingFaceFV/finevideo --repo-type dataset --include "data/train-00000-*" "data/train-00024-*" "data/train-00049-*" --local-dir /content/finevideo
!df -h /content; ls -l /content/finevideo/data

## 6. Symlinks (names stay the same, so video ids do not change). `--limit-videos` is not used

In [ ]:
%%bash
set -e
shopt -s failglob  # an unmatched glob (missing download) aborts instead of linking a literal pattern
mkdir -p /content/subset/tune /content/subset/holdout /content/ref
D=/content/finevideo/data
ln -sf $D/train-00000-*.parquet /content/subset/
ln -sf $D/train-00024-*.parquet /content/subset/
ln -sf $D/train-00049-*.parquet /content/subset/
ln -sf $D/train-00000-*.parquet /content/subset/tune/
ln -sf $D/train-00024-*.parquet /content/subset/holdout/
ln -sf $D/train-00049-*.parquet /content/subset/holdout/
for f in /content/subset/*.parquet /content/subset/tune/*.parquet /content/subset/holdout/*.parquet; do [ -e "$f" ] || { echo "broken or missing link: $f"; exit 1; }; done
ls -lR /content/subset

## 7. Clean timing run on the whole tune file (NO download running, nothing else on the GPU)
This is the plain grader command (no dumps, no checkpoints) wrapped by `bench.py`, and its output is the tune reference.
The timing must come from one uninterrupted run: if the runtime resets, just run this cell again from scratch.
The row in `bench_log.csv` must say `status=ok, clean=yes`.

In [ ]:
%%bash
cd /content/video-indexing-yolo11
PY=/content/venv/bin/python
DRIVE=/content/drive/MyDrive/vidx
mkdir -p $DRIVE/reference
$PY bench.py run --label "brute force baseline" --input /content/subset/tune --out /content/ref/ref_tune.jsonl --log $DRIVE/bench_log.csv -- --sample 0 --precision fp32
cp /content/ref/ref_tune.jsonl $DRIVE/reference/
tail -n 2 $DRIVE/bench_log.csv

## 8. Background download of all 50 files into the SAME `--local-dir` (the 3 files above are skipped)
Check the start of `download.log`: it should not re-download the 3 existing files.

In [ ]:
!nohup /content/venv/bin/hf download HuggingFaceFV/finevideo --repo-type dataset --include "data/train-000[0-4]*" --local-dir /content/finevideo > /content/download.log 2>&1 &

In [ ]:
!sleep 20; head -c 1500 /content/download.log; echo; ls /content/finevideo/data | wc -l

## 9. Tune dump run (may overlap the download; not timed)
Per-frame dumps and checkpoints go to Drive, the output stays local. Gives the data for offline tuning and a determinism check
against `ref_tune.jsonl`. After a reset: redo steps 1-4, re-download the 3 reference files (step 5), recreate the symlinks (step 6) and, if the 50-file download (step 8) had not finished, start it again, then re-run this command with `--resume`.

In [ ]:
%%bash
cd /content/video-indexing-yolo11
PY=/content/venv/bin/python
DRIVE=/content/drive/MyDrive/vidx
nohup $PY run.py --input /content/subset/tune --output /content/ref/ref_tune_dump.jsonl --dump-frames $DRIVE/frames --checkpoint-dir $DRIVE/ckpt --run-name ref_tune_dump --sample 0 --precision fp32 > /content/tune_dump.log 2>&1 &
sleep 5; tail -n 3 /content/tune_dump.log

In [ ]:
!tail -n 3 /content/tune_dump.log; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

## 10. Holdout reference (no dumps: holdout is never tuned on)
After a reset: redo steps 1-4, re-download the 3 reference files (step 5), recreate the symlinks (step 6) and, if the 50-file download (step 8) had not finished, start it again, then add `--resume` to the command.

In [ ]:
%%bash
cd /content/video-indexing-yolo11
PY=/content/venv/bin/python
DRIVE=/content/drive/MyDrive/vidx
nohup $PY run.py --input /content/subset/holdout --output /content/ref/ref_holdout.jsonl --checkpoint-dir $DRIVE/ckpt --run-name ref_holdout --sample 0 --precision fp32 > /content/holdout.log 2>&1 &
sleep 5; tail -n 3 /content/holdout.log

In [ ]:
!tail -n 3 /content/holdout.log

## 11. Checks (after both runs finish)

In [ ]:
%%bash
cd /content/video-indexing-yolo11
PY=/content/venv/bin/python
DRIVE=/content/drive/MyDrive/vidx
$PY tools/check_schema.py /content/ref/ref_tune.jsonl
$PY tools/check_schema.py /content/ref/ref_holdout.jsonl
# self-scores: expect 1.0 / 100% / 100%
$PY score.py /content/ref/ref_tune.jsonl /content/ref/ref_tune.jsonl
$PY score.py /content/ref/ref_holdout.jsonl /content/ref/ref_holdout.jsonl
# determinism: timing run vs dump run, expect about 1.0
$PY score.py /content/ref/ref_tune.jsonl /content/ref/ref_tune_dump.jsonl
$PY tools/find_rotated.py --input /content/subset/tune --max-files 1
$PY tools/decoder_agreement.py --input /content/subset/tune --n 20
$PY tools/batch_agreement.py --input /content/subset/tune --frames 1000

## 12. Copy the reference outputs to Drive

In [ ]:
%%bash
cd /content/video-indexing-yolo11
PY=/content/venv/bin/python
DRIVE=/content/drive/MyDrive/vidx
mkdir -p $DRIVE/reference
cp /content/ref/ref_tune.jsonl /content/ref/ref_tune_dump.jsonl /content/ref/ref_holdout.jsonl $DRIVE/reference/
ls -l $DRIVE/reference

## 13. After the 50-file download has finished (`tail /content/download.log`, 50 files in `data/`)
Model-free frame counts (demux only): all 50 files, then the tune file compared with its dumps (gaps must be about 0).
Compare the resolution mix of the tune file with all 50 files; if they differ a lot, say so in the notes.
Then extrapolate the clean brute-force row (labelled `extrapolated` in `bench_log.csv`).

In [ ]:
%%bash
cd /content/video-indexing-yolo11
PY=/content/venv/bin/python
DRIVE=/content/drive/MyDrive/vidx
ls /content/finevideo/data | wc -l
$PY tools/count_frames.py --input /content/finevideo/data --out $DRIVE/frame_counts.csv --workers 8
$PY tools/count_frames.py --input /content/subset/tune --out $DRIVE/frame_counts_tune.csv --compare-dumps $DRIVE/frames
$PY bench.py extrapolate --from-label "brute force baseline" --frames-csv $DRIVE/frame_counts.csv --log $DRIVE/bench_log.csv
tail -n 2 $DRIVE/bench_log.csv

## 14. Keep the evidence
Copy `$DRIVE/bench_log.csv` into the repo (`bench_log.csv`) later; it is committed as evidence for the write-up.